# Lab 5: Train a SmolVLA robot policy

**Goal:** Trace one learner from pretrained VLM source A to early robot checkpoint C1 and the continued checkpoints at steps 500, 2000, and 5000. Check the model, config, optimizer, and processor hashes. Check the train-only statistics and the seeded development results. Compare results only when the simulator reset uses an explicit seed.

**Inputs:** Pinned local VLM and demonstrations, saved checkpoint bundles, and development results. **Compute:** Local GPU for one live C2 check. **Cloud cost:** US$0 on this prepared machine. Start Jupyter from the repository root. Read the [lab guide](README.md).

## 1. Declare the learner stages

A uses the pinned SmolVLM2 source. Its visual and language weights are frozen. The action expert and the required state/action projections start with new weights. A is not a trained robot policy. C1 trains these robot weights for 100 steps. The C2 checkpoints continue from the saved optimizer and random states. They reach steps 500, 2000, and 5000. The selected C2 is the step-5000 checkpoint. The published LIBERO model B in Lab 4 is separate and has unknown training data.

In [ ]:
from pathlib import Path
import hashlib
import json
import yaml
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd().resolve()
assert (WORKSPACE / 'configs' / 'train' / 'smolvla_bowl_plate.yaml').is_file(), 'Start Jupyter from the repository root.'
DATA_ROOT = WORKSPACE / '.local-data'
config_path = WORKSPACE / 'configs' / 'train' / 'smolvla_bowl_plate.yaml'
config = yaml.safe_load(config_path.read_text())
manifest = yaml.safe_load((WORKSPACE / 'manifests' / 'smolvla_reference.yaml').read_text())
reference = manifest['learner_start']['trained_reference']
assert isinstance(reference, dict), 'The prepared C1/C2 reference is not installed.'
assert config['c1_steps'] == 100 and config['c2_steps'] == reference['c2_step'] == 5000
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == reference['config_sha256']
print('Learner VLM source:', manifest['learner_start']['repo'])
print('Published B source:', manifest['published_reference']['repo'])
print('C1 step:', config['c1_steps'], 'C2 step:', config['c2_steps'])

## 2. Audit source and trainable weights

The A audit matched every retained VLM tensor against the pinned source. Frozen weights have no training gradient. New robot weights do. This check cannot tell you whether the policy succeeds; the live evaluation checks that.

In [ ]:
audit = json.loads((WORKSPACE / 'artifacts' / 'reference' / 'smolvla' / 'learner_init_audit.json').read_text())
assert audit['matched_loaded_tensors'] == 345 and audit['missing_loaded_tensors'] == 0
for name, group in audit['groups'].items():
    print(name, group['source'], group['parameters'], 'trainable:', group['trainable_parameters'])
assert audit['groups']['vlm_backbone']['trainable_parameters'] == 0
assert audit['groups']['action_expert']['trainable_parameters'] > 0

## 3. Check the C1 → 500 → 2000 → 5000 lineage

A bundle stores the model, `config.json`, `optimizer.pt`, the processor files, and a manifest. The parent model hash links each checkpoint to its parent. The data split and the train-only statistics must match at each stage. A changed file must fail its hash check. The next cell checks the model, `config.json`, `optimizer.pt`, and every processor file against the manifest hashes. It prints one row for each link.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

lineage_paths = [reference['c1_checkpoint'], *reference['intermediate_checkpoints'], reference['c2_checkpoint']]
bundles = []
for relative_path in lineage_paths:
    path = WORKSPACE / relative_path
    bundles.append((path, json.loads((path / 'manifest.json').read_text())))
c1_path, c1 = bundles[0]
c2_path, c2 = bundles[-1]

assert [bundle['step'] for _, bundle in bundles] == [100, 500, 2000, 5000]
assert [bundle['stage'] for _, bundle in bundles] == ['c1', 'c2', 'c2', 'c2']
assert c1['model_sha256'] == reference['c1_model_sha256']
assert c2['model_sha256'] == reference['c2_model_sha256']
assert c1['parent_model_sha256'] is None
print('step | parent -> model | model, config, optimizer, processor files')
for index, (path, bundle) in enumerate(bundles):
    checks = {
        'model': file_sha256(path / 'model.safetensors') == bundle['model_sha256'],
        'config.json': file_sha256(path / 'config.json') == bundle['config_sha256'],
        'optimizer.pt': file_sha256(path / 'optimizer.pt') == bundle['optimizer_sha256'],
    }
    assert len(bundle['processor_files_sha256']) >= 4
    for filename, expected_hash in bundle['processor_files_sha256'].items():
        checks[filename] = file_sha256(path / filename) == expected_hash
    assert all(checks.values()), [name for name, ok in checks.items() if not ok]
    if index:
        assert bundle['parent_model_sha256'] == bundles[index - 1][1]['model_sha256']
    parent = (bundle['parent_model_sha256'] or 'none')[:8]
    print(f"{bundle['step']:>4} | {parent} -> {bundle['model_sha256'][:8]} | all {len(checks)} hashes match")
for key in ['split_sha256', 'training_stats_sha256', 'vlm_sha256', 'vlm_revision']:
    assert len({json.dumps(bundle[key], sort_keys=True) for _, bundle in bundles}) == 1, key
print('Same split, train-only statistics, and VLM source in all four checkpoints.')

## 4. Inspect training and its cost

The loss is a training diagnostic. It is not a task success count. Later checkpoints saw more optimizer steps than C1, so a behavior difference includes a compute difference. The course selected the final C2 step with development checks only.

In [ ]:
stages = [('C1', c1_path.parent), *[(f'C2 step {bundle["step"]}', path.parent) for path, bundle in bundles[1:]]]
fig, axis = plt.subplots(figsize=(9, 4))
for label, run_dir in stages:
    rows = [json.loads(line) for line in (run_dir / 'metrics.jsonl').read_text().splitlines()]
    axis.plot([row['step'] for row in rows], [row['loss'] for row in rows], label=label, alpha=0.7)
    result = json.loads((run_dir / 'result.json').read_text())
    print(label, 'new training seconds:', result['elapsed_s'], 'last loss:', result['last_loss'])
axis.set(xlabel='Total optimizer step', ylabel='Training loss', yscale='log')
axis.legend()
plt.show()

## 5. Compare seeded development states

The seeded evaluator calls LIBERO reset with an explicit seed for each state. The base seed is 20260930. No final state is used. Read all 20 outcomes. Keep task failures in the denominator. A policy or infrastructure error is a separate problem. These development results guide this course. They are not a final benchmark estimate.

The unseeded 500-step check used `reset(seed=None)`. It is diagnostic only. This notebook does not compare it with the seeded results. It has no seeded 500-step score.

In [ ]:
def read_result(relative_path, expected_stage, expected_step, expected_model_sha256):
    folder = WORKSPACE / relative_path
    summary = json.loads((folder / 'summary.json').read_text())
    rows = [json.loads(line) for line in (folder / 'episodes.jsonl').read_text().splitlines()]
    assert summary['state_ids'] == list(range(20))
    assert [row['initial_state_id'] for row in rows] == list(range(20))
    assert summary['partition'] == 'development' and summary['base_seed'] == reference['base_seed']
    assert [row['metrics']['seed'] for row in rows] == [reference['base_seed'] + state for state in range(20)]
    assert (summary['stage'], summary['step'], summary['model_sha256']) == (expected_stage, expected_step, expected_model_sha256)
    assert summary['infrastructure_errors'] == summary['policy_errors'] == 0
    assert summary['successes'] == sum(row['success'] for row in rows)
    return summary, rows

c1_result, c1_rows = read_result(reference['c1_development_result'], 'c1', 100, c1['model_sha256'])
c2_2000_result, c2_2000_rows = read_result(reference['c2_2000_development_result'], 'c2', 2000, bundles[2][1]['model_sha256'])
c2_result, c2_rows = read_result(reference['c2_development_result'], 'c2', 5000, c2['model_sha256'])
assert (c1_result['successes'], c2_2000_result['successes'], c2_result['successes']) == (
    reference['c1_successes'], reference['c2_2000_successes'], reference['c2_successes']) == (0, 5, 14)
curves = [('C1 step 100', c1_result, c1_rows, 'o'), ('C2 step 2000', c2_2000_result, c2_2000_rows, 's'), ('C2 step 5000', c2_result, c2_rows, 'x')]
for label, result, rows, _ in curves:
    print(label, result['successes'], '/', result['attempts'], 'timeouts:', sum(row['termination_reason'] == 'timeout' for row in rows), 'clipped values:', sum(row['metrics']['clipped_values'] for row in rows))
fig, axis = plt.subplots(figsize=(9, 3))
for offset, (label, _, rows, marker) in zip([-0.08, 0, 0.08], curves):
    axis.plot([state + offset for state in range(20)], [int(row['success']) for row in rows], marker, label=label)
axis.set(xlabel='Development state ID', ylabel='Success (1 or 0)', xticks=range(20), ylim=(-0.1, 1.1))
axis.legend()
plt.show()

## 6. Run one saved C2 policy live

This cell reloads the step-5000 C2 checkpoint and both processors. It runs one development state with its explicit reset seed and writes a new result directory. It uses the same bowl-plate success predicate. One repeat is a path check. It is not a new model-selection score.

In [ ]:
import tempfile
from vla_lab.evaluation.smolvla_learner import evaluate_learner_reference

parent = DATA_ROOT / 'runs'
parent.mkdir(parents=True, exist_ok=True)
output = Path(tempfile.mkdtemp(prefix='lab5-live-', dir=parent)) / 'result'
live = evaluate_learner_reference(
    checkpoint=c2_path,
    vlm_dir=DATA_ROOT / 'models' / 'smolvlm2-500m-video-instruct',
    configs_root=WORKSPACE / 'configs',
    data_root=DATA_ROOT,
    output_dir=output,
    state_ids=[2],
)
assert live['attempts'] == 1 and live['infrastructure_errors'] == 0
print('Live state 2:', live['successes'], '/', live['attempts'], 'result:', output)

## 7. Read the first ACT/VLA development comparison

The ACT demo checkpoint, published SmolVLA B, and trained learner C2 ran on the same 20 bowl-plate development states with explicit simulator reset seeds. This is a practical recipe comparison. ACT used 200 robot training steps and no pretrained visual weights. C2 used a pretrained VLM and 5,000 robot training steps. B has unknown training data. The paired counts show which states changed, but development data helped select the VLA settings. Use the reserved final states only after all recipes are frozen.


In [ ]:
comparison = json.loads((WORKSPACE / 'artifacts' / 'reference' / 'smolvla' / 'first_act_vla_development.json').read_text())
assert comparison['state_ids'] == list(range(20)) and comparison['base_seed'] == 20260930
assert comparison['final_states_used'] is False
for name, result in comparison['methods'].items():
    print(name, result['successes'], '/', result['attempts'], '95% Wilson:', [round(value, 3) for value in result['wilson_95']])
for pair in comparison['pairs']:
    print(pair['a'], 'vs', pair['b'], '| ACT only:', pair['a_only'], '| other only:', pair['b_only'])
print(comparison['comparison_limit'])
print(comparison['selection_limit'])


## Completion check

You are done when you can do these tasks:

- Name the loaded and new weight groups of A.
- Show the parent hashes from C1 to step 500, step 2000, and step 5000.
- Show that the model, `config.json`, `optimizer.pt`, and processor hashes match each manifest.
- Report the seeded development results: C1 0/20, C2 step 2000 5/20, and C2 step 5000 14/20.
- State why the 500-step `reset(seed=None)` check is diagnostic only.
- State why B is not a fine-tuning parent for C.

The [exercise](exercise.md) trains a 1000-step continuation from C1. The [solution guide](../../solutions/05-adapt-smolvla.md) is separate.